In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Hit:6 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [1,845 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,082 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/res

In [ ]:
from pyspark.sql import SparkSession
import pyspark.pandas as ps
from pyspark.sql.functions import desc, lit
import pandas as pd
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.functions import desc, lit, when, cast
from pyspark.sql.functions import concat_ws, col, concat, lpad, year, month, dayofmonth, sum
import pyspark.sql.functions as sf
import pyspark.sql.functions as sf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
spark = SparkSession.builder.getOrCreate()
excel_path = "/content/drive/MyDrive/Colab Notebooks/Store.xlsx"

In [ ]:
pandas_df = pd.read_excel(excel_path, sheet_name="Central_store")
central = spark.createDataFrame(pandas_df)

pandas_df = pd.read_excel(excel_path, sheet_name="West_store")
west = spark.createDataFrame(pandas_df)

pandas_df = pd.read_excel(excel_path, sheet_name="East_store")
east = spark.createDataFrame(pandas_df)

pandas_df = pd.read_excel(excel_path, sheet_name="South_store")
south = spark.createDataFrame(pandas_df)

pandas_df = pd.read_excel(excel_path, sheet_name="Managers")
managers = spark.createDataFrame(pandas_df)

pandas_df = pd.read_excel(excel_path, sheet_name="Customers Returns")
returns = spark.createDataFrame(pandas_df)

In [ ]:
spark_df = central.union(west).union(east).union(south)
# spark_df = spark_df.withColumn("Postal Code", col("Postal Code").cast("int"))
spark_df.show()

+------+--------------+-------------------+-------------------+--------------+-----------+----------------+-----------+-------------+----------+------------+-----------+-------+---------------+---------------+------------+--------------------+------------------+--------+--------+-------------------+
|Row ID|      Order ID|         Order Date|          Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|      City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|             Sales|Quantity|Discount|             Profit|
+------+--------------+-------------------+-------------------+--------------+-----------+----------------+-----------+-------------+----------+------------+-----------+-------+---------------+---------------+------------+--------------------+------------------+--------+--------+-------------------+
|     1|CA-2011-103800|2013-01-03 00:00:00|2013-01-07 00:00:00|Standard Class|   DP-13000|   Darr

In [ ]:
from pyspark.sql.functions import approx_count_distinct
unique_counts = [spark_df.agg(approx_count_distinct(col)).collect()[0][0] for col in spark_df.columns]

# Create a dictionary with column names and their corresponding unique value counts
col_unique_counts = dict(zip(spark_df.columns, unique_counts))

# Display the dictionary
print("Number of unique values in each column:")
for col, count in col_unique_counts.items():
    print(f"{col}: {count}")

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
date_df = (spark_df.select(concat(year("Ship Date"), lpad(month("Ship Date"), 2, "0"), lpad(dayofmonth("Ship Date"), 2, "0")).alias("Date_Key"), spark_df["Ship Date"].alias("DateTime"), year("Ship Date").alias("Year"), month("Ship Date").alias("Month"), dayofmonth("Ship Date").alias("Day")))\
                   .union(spark_df.select(concat(year("Ship Date"), lpad(month("Ship Date"), 2, "0"), lpad(dayofmonth("Ship Date"), 2, "0")).alias("Date_Key"), spark_df["Ship Date"].alias("DateTime"), year("Ship Date").alias("Year"), month("Ship Date").alias("Month"), dayofmonth("Ship Date").alias("Day"))).distinct()

date_df.show()

+--------+-------------------+----+-----+---+
|Date_Key|           DateTime|Year|Month|Day|
+--------+-------------------+----+-----+---+
|20130916|2013-09-16 00:00:00|2013|    9| 16|
|20150929|2015-09-29 00:00:00|2015|    9| 29|
|20160603|2016-06-03 00:00:00|2016|    6|  3|
|20160819|2016-08-19 00:00:00|2016|    8| 19|
|20161005|2016-10-05 00:00:00|2016|   10|  5|
|20130917|2013-09-17 00:00:00|2013|    9| 17|
|20140422|2014-04-22 00:00:00|2014|    4| 22|
|20140508|2014-05-08 00:00:00|2014|    5|  8|
|20150627|2015-06-27 00:00:00|2015|    6| 27|
|20130622|2013-06-22 00:00:00|2013|    6| 22|
|20140124|2014-01-24 00:00:00|2014|    1| 24|
|20151119|2015-11-19 00:00:00|2015|   11| 19|
|20160922|2016-09-22 00:00:00|2016|    9| 22|
|20131015|2013-10-15 00:00:00|2013|   10| 15|
|20140613|2014-06-13 00:00:00|2014|    6| 13|
|20150925|2015-09-25 00:00:00|2015|    9| 25|
|20151129|2015-11-29 00:00:00|2015|   11| 29|
|20130926|2013-09-26 00:00:00|2013|    9| 26|
|20140616|2014-06-16 00:00:00|2014

In [ ]:
customer_information = spark_df.select("Customer ID", "Customer Name", "Segment")\
                               .distinct()\
                               .withColumn("Customer_Key", monotonically_increasing_id())

customer_information.show()

+-----------+------------------+-----------+------------+
|Customer ID|     Customer Name|    Segment|Customer_Key|
+-----------+------------------+-----------+------------+
|   GH-14665|       Greg Hansen|   Consumer|           0|
|   PN-18775|    Parhena Norris|Home Office|           1|
|   LS-17200|      Luke Schmidt|  Corporate|           2|
|   MG-17875|     Michael Grace|Home Office|           3|
|   GZ-14470|     Gary Zandusky|   Consumer|           4|
|   DR-12880|   Dan Reichenbach|  Corporate|           5|
|   SU-20665|Stephanie Ulpright|Home Office|           6|
|   SC-20575|      Sonia Cooley|   Consumer|           7|
|   NC-18415|       Nathan Cano|   Consumer|           8|
|   DJ-13420|         Denny Joy|  Corporate|           9|
|   BK-11260|    Berenike Kampe|   Consumer|          10|
|   CS-12490|  Cindy Schnelling|  Corporate|          11|
|   NP-18700|        Nora Preis|   Consumer|          12|
|   BG-11035|    Barry Gonzalez|   Consumer|          13|
|   PW-19030| 

In [ ]:
customer_information.count()

793

In [ ]:
location_information = spark_df.select("City", "State", "Postal Code")\
                               .distinct()\
                               .withColumn("Location_Key", monotonically_increasing_id())\
                               .withColumn("Country", lit("United States"))

location_information = location_information.select("Country", "City", "State", "Postal Code", "Location_Key")

location_information.show()

+-------------+----------------+---------+-----------+------------+
|      Country|            City|    State|Postal Code|Location_Key|
+-------------+----------------+---------+-----------+------------+
|United States|          Taylor| Michigan|    48180.0|           0|
|United States|    Saint Peters| Missouri|    63376.0|           1|
|United States|       Greenwood|  Indiana|    46142.0|           2|
|United States|        Waukesha|Wisconsin|    53186.0|           3|
|United States|        Wheeling| Illinois|    60090.0|           4|
|United States|      The Colony|    Texas|    75056.0|           5|
|United States|Sterling Heights| Michigan|    48310.0|           6|
|United States|         Detroit| Michigan|    48227.0|           7|
|United States|     Port Arthur|    Texas|    77642.0|           8|
|United States|         Houston|    Texas|    77036.0|           9|
|United States|       Urbandale|     Iowa|    50322.0|          10|
|United States|         Houston|    Texas|    77

In [ ]:
location_information.count()

633

In [ ]:
category_information = spark_df.select("Category", "Sub-Category")\
                               .distinct()\
                               .withColumn("Category_Key", monotonically_increasing_id())

category_information.show()

+---------------+------------+------------+
|       Category|Sub-Category|Category_Key|
+---------------+------------+------------+
|Office Supplies|  Appliances|           0|
|Office Supplies|   Envelopes|           1|
|      Furniture|      Chairs|           2|
|Office Supplies|     Storage|           3|
|Office Supplies|       Paper|           4|
|Office Supplies|    Supplies|           5|
|     Technology|     Copiers|           6|
|Office Supplies|         Art|           7|
|Office Supplies|     Binders|           8|
|     Technology| Accessories|           9|
|Office Supplies|   Fasteners|          10|
|      Furniture|   Bookcases|          11|
|     Technology|      Phones|          12|
|Office Supplies|      Labels|          13|
|     Technology|    Machines|          14|
|      Furniture|      Tables|          15|
|      Furniture| Furnishings|          16|
+---------------+------------+------------+



In [ ]:
product_information = spark_df.select("Product ID", "Category", "Sub-Category", "Product Name")\
                              .distinct()\
                              .withColumn("Product_Key", monotonically_increasing_id())

product_information.show()

+---------------+---------------+------------+--------------------+-----------+
|     Product ID|       Category|Sub-Category|        Product Name|Product_Key|
+---------------+---------------+------------+--------------------+-----------+
|OFF-AR-10001953|Office Supplies|         Art|Boston 1645 Delux...|          0|
|TEC-PH-10001552|     Technology|      Phones|I Need's 3d Hello...|          1|
|OFF-PA-10001937|Office Supplies|       Paper|            Xerox 21|          2|
|OFF-EN-10000483|Office Supplies|   Envelopes|White Envelopes, ...|          3|
|FUR-BO-10004360|      Furniture|   Bookcases|Rush Hierlooms Co...|          4|
|TEC-AC-10001767|     Technology| Accessories|SanDisk Ultra 64 ...|          5|
|OFF-PA-10003039|Office Supplies|       Paper|          Xerox 1960|          6|
|OFF-BI-10003460|Office Supplies|     Binders|   Acco 3-Hole Punch|          7|
|TEC-AC-10001314|     Technology| Accessories|Case Logic 2.4GHz...|          8|
|TEC-PH-10000148|     Technology|      P

In [ ]:
product_df = product_information.join(category_information, on=["Category", "Sub-Category"], how="inner") \
                                .select("Product ID", "Category_Key", "Product_Key", "Product Name")

product_df.show()

+---------------+------------+-----------+--------------------+
|     Product ID|Category_Key|Product_Key|        Product Name|
+---------------+------------+-----------+--------------------+
|OFF-AP-10001124|           0|       1887|Belkin 8 Outlet S...|
|OFF-AP-10003099|           0|       1883|Eureka Hand Vacuu...|
|OFF-AP-10002734|           0|       1882|Holmes Harmony HE...|
|OFF-AP-10003914|           0|       1880|Sanitaire Vibra G...|
|OFF-AP-10003884|           0|       1875|Fellowes Smart Su...|
|OFF-AP-10000159|           0|       1841|Belkin F9M820V08 ...|
|OFF-AP-10003860|           0|       1830|Fellowes Advanced...|
|OFF-AP-10003266|           0|       1816|Holmes Replacemen...|
|OFF-AP-10002670|           0|       1769|Belkin 8-Outlet P...|
|OFF-AP-10002287|           0|       1739|Eureka Sanitaire ...|
|OFF-AP-10003622|           0|       1694|Bravo II Megaboss...|
|OFF-AP-10001469|           0|       1676|Fellowes 8 Outlet...|
|OFF-AP-10004540|           0|       167

In [ ]:
unique_counts = [product_df.agg(approx_count_distinct(col)).collect()[0][0] for col in product_df.columns]

# Create a dictionary with column names and their corresponding unique value counts
col_unique_counts = dict(zip(product_df.columns, unique_counts))

# Display the dictionary
print("Number of unique values in each column:")
for col, count in col_unique_counts.items():
    print(f"{col}: {count}")

Number of unique values in each column:
Product ID: 2002
Category_Key: 17
Product_Key: 1819
Product Name: 1802


In [ ]:
category_information.count()

17

In [ ]:
resampled_df = spark_df.join(customer_information, on=["Customer ID", "Customer Name", "Segment"], how="inner") \
                    .join(location_information, on=["Country", "City", "State", "Postal Code"], how="inner") \
                    .join(product_df, on=["Product ID", "Product Name"], how="inner") \
                    .join(date_df.withColumnsRenamed({"Date_Key": "Ship Date Key", "DateTime" : "ShipDateTime"}), on=col("Ship Date") == col("ShipDateTime"), how="inner") \
                    .join(date_df.withColumnsRenamed({"Date_Key": "Order Date Key", "DateTime" : "OrderDateTime"}), on=col("Order Date") == col("OrderDateTime"), how="inner") \
                    .select("Row ID", "Order ID", "Order Date Key", "Ship Date Key", "Ship Mode", "Region", "Customer_Key", "Location_Key", "Product_Key", "Category_Key", "Sales", "Quantity", "Discount", "Profit")

resampled_df.show()

+------+--------------+--------------+-------------+--------------+-------+------------+------------+-----------+------------+------------------+--------+--------+-------------------+
|Row ID|      Order ID|Order Date Key|Ship Date Key|     Ship Mode| Region|Customer_Key|Location_Key|Product_Key|Category_Key|             Sales|Quantity|Discount|             Profit|
+------+--------------+--------------+-------------+--------------+-------+------------+------------+-----------+------------+------------------+--------+--------+-------------------+
|  3821|CA-2015-103744|      20150223|     20150227|Standard Class|Central|           3|          99|        846|           8| 4.427999999999999|       3|     0.8| -6.863399999999999|
| 17097|CA-2018-106964|      20181217|     20181220|   First Class|   West|          22|         253|        846|           8|            11.808|       2|     0.2|             4.2804|
| 16738|CA-2014-106964|      20161217|     20161220|   First Class|   West|     

In [ ]:
full_df = resampled_df.join(returns_df.distinct(), on=["Order ID"], how="left") \
                    .withColumn("Returned", when(returns_df["Order ID"].isNotNull(), "yes").otherwise("no")) \
                    .select("Row ID", "Order ID", "Order Date Key", "Ship Date Key", "Ship Mode", "Region", "Customer_Key", "Location_Key", "Product_Key", "Category_Key", "Sales", "Quantity", "Discount", "Profit", "Returned")

full_df.show()

+------+--------------+--------------+-------------+--------------+------+------------+------------+-----------+------------+------------------+--------+--------+------------------+--------+
|Row ID|      Order ID|Order Date Key|Ship Date Key|     Ship Mode|Region|Customer_Key|Location_Key|Product_Key|Category_Key|             Sales|Quantity|Discount|            Profit|Returned|
+------+--------------+--------------+-------------+--------------+------+------------+------------+-----------+------------+------------------+--------+--------+------------------+--------+
|  4972|CA-2011-152422|      20131005|     20131010|  Second Class|  East|         775|         426|       1664|           0|             91.92|       5|     0.2|11.489999999999991|      no|
|  9203|CA-2015-152422|      20151005|     20151010|  Second Class|  East|         775|         426|       1664|           0|             91.92|       5|     0.2|11.489999999999991|      no|
|  8483|CA-2016-120810|      20160723|     20

In [ ]:
region_total_sales = full_df.join(location_information, on="Location_Key", how="inner").groupBy("Region").agg(sf.sum("Sales").alias("Total Sales")).orderBy(desc("Total Sales"))

region_total_sales.show(truncate=False)

+-------+------------------+
|Region |Total Sales       |
+-------+------------------+
|West   |1410175.0639999823|
|East   |1331287.9809999848|
|Central|966251.6575999964 |
|South  |751875.775999997  |
+-------+------------------+



In [ ]:
region_avg_sales = full_df.join(location_information, on="Location_Key", how="inner").groupBy("Region").agg(sf.avg("Sales").alias("Average Sales")).orderBy(desc("Average Sales"))

region_avg_sales.show(truncate=False)

+-------+------------------+
|Region |Average Sales     |
+-------+------------------+
|East   |240.34807384004063|
|South  |239.070199046104  |
|West   |227.08133075684083|
|Central|216.06700751341603|
+-------+------------------+



In [ ]:
category_profit = full_df.join(category_information, on="Category_Key", how="inner").groupBy("Category").agg(sf.sum("Profit").alias("Total Profit")).orderBy(desc("Total Profit"))

category_profit.show(truncate=False)

+---------------+------------------+
|Category       |Total Profit      |
+---------------+------------------+
|Technology     |281775.30449999915|
|Office Supplies|239700.94629999954|
|Furniture      |36078.10789999997 |
+---------------+------------------+



In [ ]:
subcategory_profit = full_df.join(category_information, on="Category_Key", how="inner").groupBy("Sub-Category").agg(sf.sum("Profit").alias("Total Profit")).orderBy(desc("Total Profit"))

subcategory_profit.show(truncate=False)

+------------+-------------------+
|Sub-Category|Total Profit       |
+------------+-------------------+
|Copiers     |109855.67279999993 |
|Phones      |86024.70690000005  |
|Accessories |81241.81689999958  |
|Paper       |66260.65130000006  |
|Binders     |60226.67080000018  |
|Chairs      |51247.91450000001  |
|Storage     |41701.351100000036 |
|Appliances  |34978.01009999999  |
|Furnishings |24905.62189999998  |
|Envelopes   |13668.495999999986 |
|Art         |12703.077600000002 |
|Labels      |10546.118899999989 |
|Machines    |4653.10789999998   |
|Fasteners   |1867.7174999999972 |
|Supplies    |-2251.1470000000004|
|Bookcases   |-6341.857399999998 |
|Tables      |-33733.57109999998 |
+------------+-------------------+



In [ ]:
category_total_quantity = full_df.join(product_df, on=["Category_Key"], how="inner").groupBy("Category").agg(sf.sum("Quantity").alias("Total Quantity")).orderBy(desc("Total Quantity"))

category_total_quantity.show(truncate=False)

+-----------------------------------------------------------+--------------+
|Product Name                                               |Total Quantity|
+-----------------------------------------------------------+--------------+
|Staples                                                    |412           |
|Staple envelope                                            |316           |
|Easy-staple paper                                          |293           |
|Avery Non-Stick Binders                                    |142           |
|Staples in misc. colors                                    |141           |
|Storex Dura Pro Binders                                    |139           |
|KI Adjustable-Height Table                                 |122           |
|GBC Premium Transparent Covers with Diagonal Lined Pattern |121           |
|Chromcraft Round Conference Tables                         |116           |
|Staple-based wall hangings                                 |114           |

In [ ]:
year_total_profit = full_df.join(date_df.withColumnRenamed("Date_Key", "Order Date Key"), on="Order Date Key", how="inner").filter(col("Returned") == "no").groupBy("Year").agg(sf.sum("Profit").alias("Total Profit")).orderBy("Year")

year_total_profit.show(truncate=False)

+----+------------------+
|Year|Total Profit      |
+----+------------------+
|2013|49378.67049999999 |
|2014|51479.77100000013 |
|2015|129517.80990000012|
|2016|150378.61899999998|
|2017|78633.96359999971 |
|2018|75065.99599999998 |
+----+------------------+



In [ ]:
customer_segment_avg = full_df.join(customer_information, on=["Customer_Key"], how="inner").groupBy("Segment").agg(sf.avg("Profit").alias("Avg profit per segment"))

customer_segment_avg.show(truncate=False)

+-----------+----------------------+
|Segment    |Avg profit per segment|
+-----------+----------------------+
|Consumer   |26.073575961442767    |
|Home Office|33.203956260869596    |
|Corporate  |30.85978382026322     |
+-----------+----------------------+



In [ ]:
subcategory_profit.write.json("./subcategory_profit.json", mode="overwrite")
year_total_profit.write.json("./year_total_profit.json", mode="overwrite")